In [0]:
# COMMAND ----------
# MAGIC %run ../log/00_audit_logging

# COMMAND ----------
from datetime import datetime

CATALOG = "workspace"
SCHEMA = "tmdb"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_movies"
SILVER_MOVIES = f"{CATALOG}.{SCHEMA}.silver_movies"
SILVER_GENRE_BRIDGE = f"{CATALOG}.{SCHEMA}.silver_genre_bridge"

# Fallback logger definition
if 'log_pipeline_execution' not in globals():
    def log_pipeline_execution(layer, load_type, batch_id, read_path, start_time, end_time, status, rows_inserted=0, rows_quarantined=0, error_msg=""):
        print(f"[AUDIT LOG] Layer: {layer} | Type: {load_type} | Batch: {batch_id} | Status: {status} | Inserted: {rows_inserted} | Quarantined: {rows_quarantined}")
        if error_msg:
            print(f"[AUDIT LOG ERROR] {error_msg}")

# Create Silver Delta Tables with aligned schemas
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_MOVIES} (
    movie_id          LONG,
    title             STRING,
    original_title    STRING,
    overview          STRING,
    release_date      DATE,
    popularity        DOUBLE,
    vote_average      DOUBLE,
    vote_count        LONG,
    adult             BOOLEAN,
    backdrop_path     STRING,
    poster_path       STRING,
    original_language STRING,
    source_collection STRING,
    batch_id          STRING,
    load_timestamp    TIMESTAMP,
    updated_at        TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_GENRE_BRIDGE} (
    movie_id       LONG,
    genre_id       LONG,
    genre_name     STRING,
    batch_id       STRING,
    load_timestamp TIMESTAMP,
    updated_at     TIMESTAMP
) USING DELTA
""")

In [0]:
# COMMAND ----------
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType, ArrayType
)

dbutils.widgets.text("batch_id", "", "Batch ID (blank = process latest bronze batch)")
target_batch_id = dbutils.widgets.get("batch_id").strip()

# Schema for raw_record JSON string (release_date kept as StringType to avoid parse errors)
movie_schema = StructType([
    StructField("id",                LongType(),    True),
    StructField("title",             StringType(),  True),
    StructField("original_title",    StringType(),  True),
    StructField("overview",          StringType(),  True),
    StructField("release_date",      StringType(),  True),
    StructField("popularity",        DoubleType(),  True),
    StructField("vote_average",      DoubleType(),  True),
    StructField("vote_count",        LongType(),    True),
    StructField("adult",             BooleanType(), True),
    StructField("backdrop_path",     StringType(),  True),
    StructField("poster_path",       StringType(),  True),
    StructField("original_language", StringType(),  True),
    StructField("genres", ArrayType(
        StructType([
            StructField("id",   LongType(),   True),
            StructField("name", StringType(), True)
        ])
    ), True)
])

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read Bronze Table
bronze_df = spark.table(BRONZE_TABLE)

# Identify target batch ID
if not target_batch_id:
    batch_row = bronze_df.select(F.max("batch_id")).collect()
    latest_batch_id = batch_row[0][0] if batch_row and batch_row[0][0] else "unknown_batch"
else:
    latest_batch_id = target_batch_id

# Filter Bronze records by active batch
active_bronze_df = bronze_df.filter(F.col("batch_id") == latest_batch_id)

# 2. Parse Raw JSON
parsed_bronze_df = (active_bronze_df
    .withColumn("parsed_json", F.from_json(F.col("raw_record"), movie_schema))
    .select(
        F.col("parsed_json.id").alias("movie_id"),
        F.col("parsed_json.title").alias("title"),
        F.col("parsed_json.original_title").alias("original_title"),
        F.col("parsed_json.overview").alias("overview"),
        F.col("parsed_json.release_date").alias("release_date_raw"),
        F.col("parsed_json.popularity").alias("popularity"),
        F.col("parsed_json.vote_average").alias("vote_average"),
        F.col("parsed_json.vote_count").alias("vote_count"),
        F.col("parsed_json.adult").alias("adult"),
        F.col("parsed_json.backdrop_path").alias("backdrop_path"),
        F.col("parsed_json.poster_path").alias("poster_path"),
        F.col("parsed_json.original_language").alias("original_language"),
        F.col("parsed_json.genres").alias("genres"),
        "source_collection",
        "batch_id",
        "load_timestamp"
    )
)

# 3. Clean Movies DataFrame (convert date, window dedup, add updated_at)
movie_window = Window.partitionBy("movie_id").orderBy(F.col("load_timestamp").desc())

silver_movies_df = (parsed_bronze_df
    .filter(F.col("movie_id").isNotNull())
    # Safely convert release_date_raw string to DATE (turns empty strings "" into NULL)
    .withColumn("release_date", F.try_to_date(F.col("release_date_raw")))
    .withColumn("rk", F.row_number().over(movie_window))
    .filter(F.col("rk") == 1)
    .drop("rk", "genres", "release_date_raw")
    .withColumn("updated_at", F.current_timestamp())
)

# 4. Clean Genre Bridge DataFrame (explode, dedup, add updated_at)
silver_genres_df = (parsed_bronze_df
    .select("movie_id", "batch_id", "load_timestamp", F.explode_outer("genres").alias("genre"))
    .select(
        "movie_id",
        F.col("genre.id").alias("genre_id"),
        F.col("genre.name").alias("genre_name"),
        "batch_id",
        "load_timestamp"
    )
    .filter(F.col("movie_id").isNotNull() & F.col("genre_id").isNotNull())
    .dropDuplicates(["movie_id", "genre_id"])
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
# COMMAND ----------
from delta.tables import DeltaTable

start_time = datetime.now()

try:
    # 1. Merge into Silver Movies Table
    movies_table = DeltaTable.forName(spark, SILVER_MOVIES)
    (movies_table.alias("target")
      .merge(
        silver_movies_df.alias("source"),
        "target.movie_id = source.movie_id"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())

    # 2. Merge into Silver Genre Bridge Table
    genres_bridge_table = DeltaTable.forName(spark, SILVER_GENRE_BRIDGE)
    (genres_bridge_table.alias("target")
      .merge(
        silver_genres_df.alias("source"),
        "target.movie_id = source.movie_id AND target.genre_id = source.genre_id"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())

    movie_count = spark.table(SILVER_MOVIES).count()
    bridge_count = spark.table(SILVER_GENRE_BRIDGE).count()

    # Log execution using positional arguments
    if 'log_pipeline_execution' in globals():
        log_pipeline_execution(
            "Bronze-to-Silver",
            "delta_parse",
            latest_batch_id,
            BRONZE_TABLE,
            start_time,
            datetime.now(),
            "SUCCESS",
            movie_count,
            0
        )

    print(f"Silver Layer Pipeline Execution Succeeded!")
    print(f"Batch Processed: {latest_batch_id}")
    print(f"Total Movies in Silver: {movie_count}")
    print(f"Total Movie-Genre Mappings: {bridge_count}")

except Exception as e:
    if 'log_pipeline_execution' in globals():
        log_pipeline_execution(
            "Bronze-to-Silver",
            "delta_parse",
            latest_batch_id,
            BRONZE_TABLE,
            start_time,
            datetime.now(),
            "FAILURE",
            0,
            0,
            str(e)
        )
    raise e

In [0]:
%sql
SELECT 'silver_movies' AS table_name, COUNT(*) AS record_count FROM workspace.tmdb.silver_movies
UNION ALL
SELECT 'silver_genre_bridge' AS table_name, COUNT(*) AS record_count FROM workspace.tmdb.silver_genre_bridge
UNION ALL
SELECT 'silver_genres' AS table_name, COUNT(*) AS record_count FROM workspace.tmdb.silver_genres;

In [0]:
%sql
SELECT 
    source_collection,
    batch_id,
    COUNT(movie_id) AS total_movies,
    MIN(release_date) AS earliest_release,
    MAX(release_date) AS latest_release
FROM workspace.tmdb.silver_movies
GROUP BY source_collection, batch_id
ORDER BY source_collection, batch_id;

In [0]:
%sql
SELECT 
    m.movie_id,
    m.title,
    m.release_date,
    m.vote_average,
    g.genre_name
FROM workspace.tmdb.silver_movies m
JOIN workspace.tmdb.silver_genre_bridge b ON m.movie_id = b.movie_id
JOIN workspace.tmdb.silver_genres g ON b.genre_id = g.genre_id
ORDER BY m.popularity DESC
LIMIT 20;

In [0]:
%sql
SELECT 
    g.genre_name,
    COUNT(DISTINCT m.movie_id) AS total_movies,
    ROUND(AVG(m.vote_average), 2) AS avg_rating,
    ROUND(AVG(m.popularity), 2) AS avg_popularity
FROM workspace.tmdb.silver_movies m
JOIN workspace.tmdb.silver_genre_bridge b ON m.movie_id = b.movie_id
JOIN workspace.tmdb.silver_genres g ON b.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY total_movies DESC;

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT movie_id) AS unique_movie_ids,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_titles,
    SUM(CASE WHEN release_date IS NULL THEN 1 ELSE 0 END) AS null_release_dates
FROM workspace.tmdb.silver_movies;